# ALPA-Net No-Attention Transfer Fine-Tuning CV


## 1. Imports and CONFIG


In [1]:
import json
import logging
import random
import time
import traceback
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, balanced_accuracy_score, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
warnings.filterwarnings('ignore', category=DeprecationWarning)
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

CONFIG = {
    'SEED': 42,
    'DATASET_DIR': str(PROJECT_ROOT / 'dataset' / 'ptb_diagnostic'),
    'PRETRAIN_ROOT': str(PROJECT_ROOT / 'outputs' / '4_alpanet_pretrain_no_attention'),
    'PRETRAIN_RUN_ID': 'latest',
    'OUTPUT_BASE_DIR': str(PROJECT_ROOT / 'outputs' / '5_alpanet_finetune_no_attention'),
    'RUN_NAME': RUN_TIMESTAMP,
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
    'N_FOLDS': 5,
    'STRATEGIES': ['no_pretrain', 'frozen_backbone', 'partial_finetune', 'full_finetune'],
    'FAST_DEV_RUN': False,
    'RESUME': False,
    'num_workers': 2,
    'batch_size': 64,
    'epochs': 50,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'scheduler_patience': 5,
    'early_stopping_enabled': False,
    'early_stopping_patience': 12,
    'loss_function': 'CrossEntropyLoss',
    'classification_head': 'softmax',
    'attention_prior_strength': 0.0,
    'attention_prior_learnable': False,
    'model': {
        'input_leads': 12,
        'signal_length': 65,
        'stem_channels': 32,
        'cnn_channels': 64,
        'token_dim': 128,
        'num_heads': 4,
        'cross_lead_layers': 2,
        'temporal_layers': 1,
        'transformer_ff_dim': 256,
        'dropout': 0.20,
        'temporal_segments': 20,
        'main_outputs': 4,
        'sub_outputs': 0,
    },
}

MAIN_LABELS = ['Normal', 'Anterior', 'Inferior', 'Lateral']
MAIN_LABEL_DISPLAY = ['NORM', 'AMI', 'IMI', 'LMI']
MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
INDEX_TO_MAIN_LABEL = {idx: label for label, idx in MAIN_LABEL_TO_INDEX.items()}
LEAD_ORDER = ['I', 'aVL', 'II', 'III', 'aVF', 'aVR', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
MAIN_CLASS_LEAD_PRIORS = {
    'Normal': LEAD_ORDER,
    'Anterior': ['V1', 'V2', 'V3', 'V4'],
    'Inferior': ['II', 'III', 'aVF'],
    'Lateral': ['I', 'aVL', 'V5', 'V6'],
}

def build_main_class_prior_matrix(active_weight=0.90):
    matrix=[]; n=len(LEAD_ORDER)
    for class_name in MAIN_LABELS:
        if class_name == 'Normal':
            weights=np.ones(n,dtype=np.float32)/n
        else:
            active=MAIN_CLASS_LEAD_PRIORS[class_name]
            weights=np.full(n,(1-active_weight)/max(n-len(active),1),dtype=np.float32)
            for lead in active: weights[LEAD_ORDER.index(lead)] = active_weight/len(active)
            weights=weights/weights.sum()
        matrix.append(weights)
    return np.stack(matrix).astype(np.float32)
MAIN_CLASS_PRIOR_MATRIX = build_main_class_prior_matrix()

LABEL_MAPPING = {
    'model_name': 'ALPA-Net No-Attention Ablation',
    'task_type': 'single_label_4_class_softmax',
    'main_label_order': MAIN_LABELS,
    'main_label_display': MAIN_LABEL_DISPLAY,
    'class_to_index': MAIN_LABEL_TO_INDEX,
    'index_to_class': INDEX_TO_MAIN_LABEL,
    'head_activation_for_inference': 'softmax',
    'recommended_loss': 'CrossEntropyLoss',
}
LEAD_PRIOR_CONFIG = {
    'lead_order': LEAD_ORDER,
    'main_label_order': MAIN_LABELS,
    'main_class_lead_priors': MAIN_CLASS_LEAD_PRIORS,
    'main_class_prior_matrix': MAIN_CLASS_PRIOR_MATRIX.tolist(),
    'attention_prior_strength': CONFIG['attention_prior_strength'],
    'attention_prior_learnable': CONFIG['attention_prior_learnable'],
}
print('Device:', CONFIG['DEVICE'])
print('Dataset:', CONFIG['DATASET_DIR'])


Device: cuda
Dataset: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/dataset/ptb_diagnostic


## 2. Output Directory and Logging


In [2]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(CONFIG['SEED'])

OUTPUT_DIR = Path(CONFIG['OUTPUT_BASE_DIR']) / CONFIG['RUN_NAME']
CONFIG_DIR = OUTPUT_DIR / 'configs'; LOG_DIR = OUTPUT_DIR / 'logs'; AGG_DIR = OUTPUT_DIR / 'aggregate_results'; PLOTS_DIR = OUTPUT_DIR / 'plots'; ROOT_METRICS_DIR = OUTPUT_DIR / 'metrics'; STRATEGY_ROOT = OUTPUT_DIR / 'transfer_strategies'
for d in [CONFIG_DIR, LOG_DIR, AGG_DIR, PLOTS_DIR, ROOT_METRICS_DIR, STRATEGY_ROOT]: d.mkdir(parents=True, exist_ok=True)
for strategy in CONFIG['STRATEGIES']: (STRATEGY_ROOT / strategy).mkdir(parents=True, exist_ok=True)
with open(CONFIG_DIR/'config.json','w') as f: json.dump(CONFIG,f,indent=2)
with open(CONFIG_DIR/'label_mapping.json','w') as f: json.dump(LABEL_MAPPING,f,indent=2)
with open(CONFIG_DIR/'lead_prior_config.json','w') as f: json.dump(LEAD_PRIOR_CONFIG,f,indent=2)

def make_logger(name, train_path, error_path):
    logger=logging.getLogger(name); logger.setLevel(logging.INFO); logger.handlers.clear()
    fmt=logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
    sh=logging.StreamHandler(); sh.setFormatter(fmt)
    fh=logging.FileHandler(train_path); fh.setFormatter(fmt)
    eh=logging.FileHandler(error_path); eh.setLevel(logging.ERROR); eh.setFormatter(fmt)
    logger.addHandler(sh); logger.addHandler(fh); logger.addHandler(eh)
    return logger

global_logger = make_logger('ALPA-Net softmax transfer', LOG_DIR/'train.log', LOG_DIR/'error.log')
ROOT_OUTPUT_LOG = PROJECT_ROOT / 'output.log'
root_handler = logging.FileHandler(ROOT_OUTPUT_LOG); root_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
global_logger.addHandler(root_handler)
RUN_STARTED_AT=datetime.now(); RUN_START_TIME=time.time()
global_logger.info('='*88)
global_logger.info('ALPA-Net softmax transfer CV logger')
global_logger.info('Run started at: %s', RUN_STARTED_AT.strftime('%Y-%m-%d %H:%M:%S'))
global_logger.info('Output directory: %s', OUTPUT_DIR)
global_logger.info('Transfer strategy directory: %s', STRATEGY_ROOT)
global_logger.info('Dataset directory: %s', CONFIG['DATASET_DIR'])
global_logger.info('Device: %s', CONFIG['DEVICE'])
global_logger.info('Strategies: %s', CONFIG['STRATEGIES'])
global_logger.info('='*88)


2026-06-11 20:32:27,870 | INFO | ========================================================================================


2026-06-11 20:32:27,870 | INFO | ALPA-Net softmax transfer CV logger


2026-06-11 20:32:27,870 | INFO | Run started at: 2026-06-11 20:32:27


2026-06-11 20:32:27,871 | INFO | Output directory: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/5_alpanet_finetune_no_attention/20260611_203227


2026-06-11 20:32:27,871 | INFO | Transfer strategy directory: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/5_alpanet_finetune_no_attention/20260611_203227/transfer_strategies


2026-06-11 20:32:27,871 | INFO | Dataset directory: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/dataset/ptb_diagnostic


2026-06-11 20:32:27,871 | INFO | Device: cuda


2026-06-11 20:32:27,871 | INFO | Strategies: ['no_pretrain', 'frozen_backbone', 'partial_finetune', 'full_finetune']


2026-06-11 20:32:27,872 | INFO | ========================================================================================


## 3. Load Pretrain Source


In [3]:
def load_json(path):
    with open(path) as f: return json.load(f)

REQUIRED_PRETRAIN_FILES = [
    'alpanet_full_model.pt', 'alpanet_backbone.pt', 'alpanet_heads.pt',
    'model_config.json', 'label_mapping.json', 'lead_prior_config.json',
]

def pretrain_run_is_complete(run_dir):
    transfer = Path(run_dir) / 'transfer_ready'
    return transfer.exists() and all((transfer / name).exists() for name in REQUIRED_PRETRAIN_FILES)

def timestamp_dirs(root):
    root=Path(root)
    return sorted([p for p in root.iterdir() if p.is_dir() and pretrain_run_is_complete(p)], key=lambda p:p.name)

def load_pretrain_run(root, run_id):
    root=Path(root)
    if run_id == 'latest':
        dirs=timestamp_dirs(root)
        if not dirs: raise FileNotFoundError(f'No pretrain runs found under {root}')
        run_dir=dirs[-1]
    else:
        run_dir=root/run_id
    transfer=run_dir/'transfer_ready'
    files={
        'run_dir': run_dir,
        'transfer_dir': transfer,
        'full_model': transfer/'alpanet_full_model.pt',
        'backbone': transfer/'alpanet_backbone.pt',
        'heads': transfer/'alpanet_heads.pt',
        'model_config': transfer/'model_config.json',
        'label_mapping': transfer/'label_mapping.json',
        'lead_prior_config': transfer/'lead_prior_config.json',
    }
    missing=[str(v) for k,v in files.items() if k not in {'run_dir','transfer_dir'} and not Path(v).exists()]
    if missing: raise FileNotFoundError('Missing pretrain artifacts:\n'+'\n'.join(missing))
    return files

try:
    PRETRAIN_SOURCE=load_pretrain_run(CONFIG['PRETRAIN_ROOT'], CONFIG['PRETRAIN_RUN_ID'])
    pretrain_label_mapping=load_json(PRETRAIN_SOURCE['label_mapping'])
    if pretrain_label_mapping.get('task_type') != 'single_label_4_class_softmax':
        raise ValueError(f"Pretrain task_type incompatible: {pretrain_label_mapping.get('task_type')}")
    pretrain_main_order = pretrain_label_mapping.get('main_label_order') or pretrain_label_mapping.get('main_labels')
    if pretrain_main_order != MAIN_LABELS:
        raise ValueError(f'Pretrain main label order mismatch. Expected {MAIN_LABELS}, got {pretrain_main_order}')
    with open(CONFIG_DIR/'pretrain_source.json','w') as f: json.dump({k:str(v) for k,v in PRETRAIN_SOURCE.items()},f,indent=2)
    global_logger.info('Using pretrained source: %s', PRETRAIN_SOURCE['run_dir'])
except Exception:
    global_logger.exception('Failed to load pretrain source.')
    raise


2026-06-11 20:32:27,877 | INFO | Using pretrained source: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/4_alpanet_pretrain_no_attention/20260610_230702


## 4. Data Loading and Label Mapping


In [4]:
def load_split(dataset_dir, split):
    split_dir=Path(dataset_dir)/split
    x_path=split_dir/'x_beats.npy'
    if not x_path.exists(): x_path=split_dir/'x.npy'
    x=np.load(x_path).astype(np.float32)
    csv_path=split_dir/'metadata.csv' if (split_dir/'metadata.csv').exists() else split_dir/'labels.csv'
    labels=pd.read_csv(csv_path)
    labels['source_split']=split
    return x, labels

def parse_vector_column(value):
    import ast
    if isinstance(value, np.ndarray): return value.astype(np.float32)
    if isinstance(value, list): return np.asarray(value,dtype=np.float32)
    return np.asarray(ast.literal_eval(str(value)), dtype=np.float32)

def build_class_targets(labels_df):
    labels_df=labels_df.copy().reset_index(drop=True)
    if 'main_label_name' in labels_df.columns:
        y=labels_df['main_label_name'].map(MAIN_LABEL_TO_INDEX).to_numpy(dtype=np.int64)
    elif 'main_label_vector' in labels_df.columns:
        mat=np.vstack(labels_df['main_label_vector'].map(parse_vector_column).to_numpy())
        y=np.argmax(mat,axis=1).astype(np.int64)
        labels_df['main_label_name']=[INDEX_TO_MAIN_LABEL[int(i)] for i in y]
    else:
        raise ValueError('main_label_name or main_label_vector is required')
    labels_df['class_index']=y
    return y, labels_df

DATASET_DIR=Path(CONFIG['DATASET_DIR'])
x_train, labels_train = load_split(DATASET_DIR,'train')
x_val, labels_val = load_split(DATASET_DIR,'val')
x_test, labels_test = load_split(DATASET_DIR,'test')
y_train, labels_train = build_class_targets(labels_train)
y_val, labels_val = build_class_targets(labels_val)
y_test, labels_test = build_class_targets(labels_test)

x_all=np.concatenate([x_train,x_val,x_test],axis=0)
labels_all=pd.concat([labels_train,labels_val,labels_test],ignore_index=True)
y_all=np.concatenate([y_train,y_val,y_test])
TRAINVAL_INDICES=np.arange(len(x_train)+len(x_val))
TEST_INDICES=np.arange(len(x_train)+len(x_val), len(x_all))

global_logger.info('Loaded beat data train=%s val=%s test=%s', x_train.shape, x_val.shape, x_test.shape)
for split_name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    counts=pd.Series(y).value_counts().reindex(range(4),fill_value=0); counts.index=MAIN_LABELS
    global_logger.info('%s class distribution: %s', split_name, counts.to_dict())
    display(counts.rename(split_name).to_frame())
display(labels_all[[c for c in ['beat_id','record_id','main_label_name','class_index','source_split'] if c in labels_all.columns]].head(10))


2026-06-11 20:32:27,901 | INFO | Loaded beat data train=(1720, 65, 12) val=(224, 65, 12) test=(499, 65, 12)


2026-06-11 20:32:27,901 | INFO | train class distribution: {'Normal': 552, 'Anterior': 344, 'Inferior': 669, 'Lateral': 155}


,train
Normal,552
Anterior,344
Inferior,669
Lateral,155


2026-06-11 20:32:27,905 | INFO | val class distribution: {'Normal': 75, 'Anterior': 60, 'Inferior': 89, 'Lateral': 0}


,val
Normal,75
Anterior,60
Inferior,89
Lateral,0


2026-06-11 20:32:27,906 | INFO | test class distribution: {'Normal': 135, 'Anterior': 97, 'Inferior': 184, 'Lateral': 83}


,test
Normal,135
Anterior,97
Inferior,184
Lateral,83


,beat_id,record_id,main_label_name,class_index,source_split
0,s0398lre_beat0,s0398lre,Inferior,2,train
1,s0398lre_beat1,s0398lre,Inferior,2,train
2,s0398lre_beat2,s0398lre,Inferior,2,train
3,s0398lre_beat3,s0398lre,Inferior,2,train
4,s0398lre_beat4,s0398lre,Inferior,2,train
5,s0398lre_beat5,s0398lre,Inferior,2,train
6,s0398lre_beat6,s0398lre,Inferior,2,train
7,s0398lre_beat7,s0398lre,Inferior,2,train
8,s0398lre_beat8,s0398lre,Inferior,2,train
9,s0398lre_beat9,s0398lre,Inferior,2,train


## 5. Dataset and DataLoader


In [5]:
class PerLeadZScore:
    def __init__(self, eps=1e-6): self.eps=eps; self.mean_=None; self.std_=None
    def fit(self,x):
        self.mean_=x.mean(axis=(0,1),keepdims=True); self.std_=np.maximum(x.std(axis=(0,1),keepdims=True), self.eps); return self
    def transform(self,x): return ((x-self.mean_)/self.std_).astype(np.float32)

mean_path=PRETRAIN_SOURCE['transfer_dir']/'normalizer_mean.npy'
std_path=PRETRAIN_SOURCE['transfer_dir']/'normalizer_std.npy'
if mean_path.exists() and std_path.exists():
    normalizer=PerLeadZScore(); normalizer.mean_=np.load(mean_path); normalizer.std_=np.load(std_path)
    global_logger.info('Loaded pretrain normalizer from transfer_ready.')
else:
    normalizer=PerLeadZScore().fit(x_all[TRAINVAL_INDICES])
    global_logger.warning('Pretrain normalizer not found; fitted from target train+val pool.')
x_all_n=normalizer.transform(x_all)

class ECGClassificationDataset(Dataset):
    def __init__(self,x,y,indices):
        self.x=torch.tensor(x[indices],dtype=torch.float32); self.y=torch.tensor(y[indices],dtype=torch.long); self.indices=np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self,idx): return {'x':self.x[idx], 'y':self.y[idx], 'idx':int(self.indices[idx])}

def make_loaders(train_idx,val_idx,test_idx):
    tr=ECGClassificationDataset(x_all_n,y_all,train_idx); va=ECGClassificationDataset(x_all_n,y_all,val_idx); te=ECGClassificationDataset(x_all_n,y_all,test_idx)
    return (
        DataLoader(tr,batch_size=CONFIG['batch_size'],shuffle=True,num_workers=CONFIG['num_workers'],pin_memory=torch.cuda.is_available()),
        DataLoader(va,batch_size=CONFIG['batch_size'],shuffle=False,num_workers=CONFIG['num_workers'],pin_memory=torch.cuda.is_available()),
        DataLoader(te,batch_size=CONFIG['batch_size'],shuffle=False,num_workers=CONFIG['num_workers'],pin_memory=torch.cuda.is_available()),
    )


2026-06-11 20:32:27,914 | INFO | Loaded pretrain normalizer from transfer_ready.


## 6. ALPA-Net Softmax Model Definition


In [6]:
global_logger.info('ALPA-Net softmax model definition loaded.')

class SharedLeadCNNStem(nn.Module):
    def __init__(self, in_channels=1, stem_channels=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, stem_channels, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(stem_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MultiScaleResidualCNNBlock(nn.Module):
    def __init__(self, channels, kernels=(3, 7, 15, 31), dropout=0.1):
        super().__init__()
        branch_channels = channels // len(kernels)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(channels, branch_channels, kernel_size=k, padding=k // 2, bias=False),
                nn.BatchNorm1d(branch_channels),
                nn.GELU(),
            )
            for k in kernels
        ])
        self.project = nn.Sequential(
            nn.Conv1d(branch_channels * len(kernels), channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.Dropout(dropout),
        )
        self.act = nn.GELU()

    def forward(self, x):
        out = torch.cat([branch(x) for branch in self.branches], dim=1)
        out = self.project(out)
        return self.act(out + x)


class LeadTokenBuilder(nn.Module):
    def __init__(self, in_channels, token_dim):
        super().__init__()
        self.proj = nn.Linear(in_channels, token_dim)

    def forward(self, lead_features):
        pooled = lead_features.mean(dim=-1)
        return self.proj(pooled)




class LeadTimeAttentionPooling(nn.Module):
    def __init__(self, token_dim, dropout=0.1):
        super().__init__()
        self.score = nn.Sequential(
            nn.LayerNorm(token_dim),
            nn.Linear(token_dim, token_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(token_dim // 2, 1),
        )

    def forward(self, tokens):
        scores = self.score(tokens).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.einsum('bn,bnd->bd', weights, tokens)
        return pooled, weights


class ALPANetBackbone(nn.Module):
    def __init__(self, config):
        super().__init__()
        m = config['model']
        self.input_leads = m['input_leads']
        self.temporal_segments = m['temporal_segments']
        self.stem = SharedLeadCNNStem(1, m['stem_channels'], m['dropout'])
        self.channel_project = nn.Sequential(
            nn.Conv1d(m['stem_channels'], m['cnn_channels'], kernel_size=1, bias=False),
            nn.BatchNorm1d(m['cnn_channels']),
            nn.GELU(),
        )
        self.multi_scale = MultiScaleResidualCNNBlock(m['cnn_channels'], dropout=m['dropout'])
        self.token_builder = LeadTokenBuilder(m['cnn_channels'], m['token_dim'])
        cross_layer = nn.TransformerEncoderLayer(
            d_model=m['token_dim'], nhead=m['num_heads'], dim_feedforward=m['transformer_ff_dim'],
            dropout=m['dropout'], batch_first=True, activation='gelu', norm_first=True,
        )
        self.cross_lead_transformer = nn.TransformerEncoder(cross_layer, num_layers=m['cross_lead_layers'])
        temporal_layer = nn.TransformerEncoderLayer(
            d_model=m['token_dim'], nhead=m['num_heads'], dim_feedforward=m['transformer_ff_dim'],
            dropout=m['dropout'], batch_first=True, activation='gelu', norm_first=True,
        )
        self.temporal_transformer = nn.TransformerEncoder(temporal_layer, num_layers=m['temporal_layers'])
        self.pooling = LeadTimeAttentionPooling(m['token_dim'], dropout=m['dropout'])
        self.norm = nn.LayerNorm(m['token_dim'] * 2)

    def forward(self, x):
        b, t, l = x.shape
        x_lead = x.permute(0, 2, 1).reshape(b * l, 1, t)
        features = self.stem(x_lead)
        features = self.channel_project(features)
        features = self.multi_scale(features)
        lead_features = features.reshape(b, l, features.shape[1], features.shape[2])
        lead_tokens_raw = self.token_builder(lead_features)
        class_count = len(MAIN_LABELS)
        class_attention = lead_tokens_raw.new_full((b, class_count, l), 1.0 / float(l))
        class_context = torch.einsum('bql,bld->bqd', class_attention, lead_tokens_raw)
        lead_attention = class_attention.mean(dim=1)
        lead_tokens = self.cross_lead_transformer(lead_tokens_raw)

        segment_features = F.adaptive_avg_pool1d(features, self.temporal_segments)
        segment_features = segment_features.reshape(b, l, segment_features.shape[1], self.temporal_segments).mean(dim=1)
        temporal_tokens = segment_features.permute(0, 2, 1)
        temporal_tokens = self.temporal_transformer(self.token_builder.proj(temporal_tokens))

        lead_pooled, lead_pool_weights = self.pooling(lead_tokens)
        time_pooled, time_pool_weights = self.pooling(temporal_tokens)
        pooled_features = self.norm(torch.cat([lead_pooled, time_pooled], dim=-1))
        return {
            'pooled_features': pooled_features,
            'lead_tokens': lead_tokens,
            'temporal_tokens': temporal_tokens,
            'class_context': class_context,
            'class_attention': class_attention,
            'lead_attention': lead_attention,
            'lead_pool_weights': lead_pool_weights,
            'time_pool_weights': time_pool_weights,
        }


class ALPANet(nn.Module):
    def __init__(self, config):
        super().__init__()
        m = config['model']
        self.backbone = ALPANetBackbone(config)
        self.classification_head = nn.Sequential(
            nn.LayerNorm(m['token_dim'] * 2),
            nn.Linear(m['token_dim'] * 2, m['token_dim']),
            nn.GELU(),
            nn.Dropout(m['dropout']),
            nn.Linear(m['token_dim'], m['main_outputs']),
        )

    def forward(self, x):
        backbone_out = self.backbone(x)
        logits = self.classification_head(backbone_out['pooled_features'])
        return {
            'logits': logits,
            'probabilities': torch.softmax(logits, dim=1),
            **backbone_out,
        }


def freeze_for_transfer(model, scheme):
    for param in model.parameters():
        param.requires_grad = True
    if scheme == 'frozen_backbone':
        for param in model.backbone.parameters():
            param.requires_grad = False
    elif scheme == 'partial_finetune':
        for module in [model.backbone.stem, model.backbone.channel_project, model.backbone.multi_scale]:
            for param in module.parameters():
                param.requires_grad = False
    elif scheme in {'full_finetune', 'no_pretrain'}:
        pass
    else:
        raise ValueError(f'Unknown transfer scheme: {scheme}')


def backbone_state_dict(model):
    return model.backbone.state_dict()


def heads_state_dict(model):
    return {'classification_head': model.classification_head.state_dict()}

def build_alpanet_model():
    return ALPANet(CONFIG).to(CONFIG['DEVICE'])

global_logger.info('ALPA-Net softmax model definition completed.')


2026-06-11 20:32:27,931 | INFO | ALPA-Net softmax model definition loaded.


2026-06-11 20:32:27,932 | INFO | ALPA-Net softmax model definition completed.


## 7. Transfer Strategy Utilities


In [7]:
def load_pretrained_weights(model, strategy, logger):
    if strategy == 'no_pretrain':
        logger.info('No pretrained weights loaded.'); return model
    if strategy == 'full_finetune':
        payload=torch.load(PRETRAIN_SOURCE['full_model'], map_location=CONFIG['DEVICE'])
        model.load_state_dict(payload['model_state_dict'], strict=True)
        logger.info('Loaded pretrained full model.'); return model
    payload=torch.load(PRETRAIN_SOURCE['backbone'], map_location=CONFIG['DEVICE'])
    model.backbone.load_state_dict(payload['backbone_state_dict'], strict=True)
    logger.info('Loaded pretrained backbone.'); return model

def apply_transfer_strategy(model, strategy):
    for p in model.parameters(): p.requires_grad=True
    if strategy == 'frozen_backbone':
        for p in model.backbone.parameters(): p.requires_grad=False
    elif strategy == 'partial_finetune':
        for module in [model.backbone.stem, model.backbone.channel_project, model.backbone.multi_scale]:
            for p in module.parameters(): p.requires_grad=False
    elif strategy in {'no_pretrain','full_finetune'}:
        pass
    else: raise ValueError(strategy)
    return model

def print_trainable_parameters(model, logger):
    total=sum(p.numel() for p in model.parameters()); trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info('Parameters trainable=%d total=%d ratio=%.4f', trainable, total, trainable/max(total,1))
    return {'trainable_params':trainable,'total_params':total,'trainable_ratio':trainable/max(total,1)}

def heads_state_dict(model): return {'classification_head': model.classification_head.state_dict()}


## 8. Metrics and Evaluation Helpers


In [8]:
def softmax_np(logits):
    logits=logits-logits.max(axis=1,keepdims=True); exp=np.exp(logits); return exp/exp.sum(axis=1,keepdims=True)

def compute_metrics(y_true, logits):
    prob=softmax_np(logits); pred=np.argmax(prob,axis=1)
    cm=confusion_matrix(y_true,pred,labels=np.arange(len(MAIN_LABELS)))
    out={
        'accuracy':float(accuracy_score(y_true,pred)),
        'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),
        'macro_f1':float(f1_score(y_true,pred,average='macro',zero_division=0)),
        'micro_f1':float(f1_score(y_true,pred,average='micro',zero_division=0)),
        'weighted_f1':float(f1_score(y_true,pred,average='weighted',zero_division=0)),
    }
    per=f1_score(y_true,pred,labels=np.arange(len(MAIN_LABELS)),average=None,zero_division=0)
    class_rows=[]; auc_values=[]
    for idx,(label,display_label,val) in enumerate(zip(MAIN_LABELS,MAIN_LABEL_DISPLAY,per)):
        tp=float(cm[idx,idx]); fn=float(cm[idx,:].sum()-cm[idx,idx]); fp=float(cm[:,idx].sum()-cm[idx,idx]); tn=float(cm.sum()-tp-fn-fp)
        sensitivity=tp/max(tp+fn,1.0); specificity=tn/max(tn+fp,1.0)
        try:
            auc_value=float(roc_auc_score((y_true==idx).astype(int), prob[:,idx])); auc_values.append(auc_value)
        except ValueError:
            auc_value=np.nan
        out[f'f1_{label}']=float(val); out[f'sensitivity_{label}']=float(sensitivity); out[f'specificity_{label}']=float(specificity); out[f'auc_{label}']=float(auc_value) if np.isfinite(auc_value) else np.nan
        class_rows.append({'label':display_label,'internal_label':label,'f1_score':float(val),'sensitivity':float(sensitivity),'specificity':float(specificity),'auc':float(auc_value) if np.isfinite(auc_value) else np.nan,'support':int((y_true==idx).sum())})
    out['macro_auc']=float(np.nanmean(auc_values)) if auc_values else np.nan
    return out, prob, pred, pd.DataFrame(class_rows)

def run_epoch(model, loader, criterion, optimizer=None):
    training=optimizer is not None; model.train(training)
    total=0; n=0; logits=[]; y=[]; idxs=[]; attn=[]
    ctx=torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for b in loader:
            xb=b['x'].to(CONFIG['DEVICE']); yb=b['y'].to(CONFIG['DEVICE'])
            if training: optimizer.zero_grad(set_to_none=True)
            out=model(xb); loss=criterion(out['logits'], yb)
            if training:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            bs=xb.size(0); total += float(loss.detach().cpu())*bs; n+=bs
            logits.append(out['logits'].detach().cpu().numpy()); y.append(yb.detach().cpu().numpy()); idxs.append(b['idx'].detach().cpu().numpy())
            if 'class_attention' in out: attn.append(out['class_attention'].detach().cpu().numpy())
    state={'loss':total/max(n,1),'logits':np.concatenate(logits),'y_true':np.concatenate(y),'idx':np.concatenate(idxs)}
    if attn: state['class_attention']=np.concatenate(attn)
    return state

def save_cm(y_true,y_pred,labels,path_png,path_csv,title,metrics_dir=None):
    cm=confusion_matrix(y_true,y_pred,labels=np.arange(len(labels)))
    pd.DataFrame(cm,index=labels,columns=labels).to_csv(path_csv)
    fig,ax=plt.subplots(figsize=(9,8),dpi=400)
    sns.heatmap(cm,annot=True,fmt='d',cmap='Oranges',xticklabels=labels,yticklabels=labels,linewidths=1,linecolor='black',ax=ax,annot_kws={'fontsize':14})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title,fontsize=16,weight='bold')
    fig.tight_layout(); fig.savefig(path_png,bbox_inches='tight')
    if metrics_dir is not None:
        metrics_dir=Path(metrics_dir); metrics_dir.mkdir(parents=True,exist_ok=True)
        pd.DataFrame(cm,index=labels,columns=labels).to_csv(metrics_dir/path_csv.name)
        fig.savefig(metrics_dir/path_png.name,bbox_inches='tight')
    plt.close(fig)
    return cm

def save_roc_curves(y_true, prob, output_png, output_csv, title, metrics_png=None):
    rows=[]; fig,ax=plt.subplots(figsize=(9,7),dpi=400)
    for idx,label in enumerate(MAIN_LABEL_DISPLAY):
        binary=(y_true==idx).astype(int)
        if binary.min()==binary.max():
            continue
        fpr,tpr,_=roc_curve(binary,prob[:,idx]); auc_value=roc_auc_score(binary,prob[:,idx])
        ax.plot(fpr,tpr,linewidth=2.2,label=f'{label} AUC={auc_value:.3f}')
        rows.extend([{'label':label,'fpr':float(x),'tpr':float(y),'auc':float(auc_value)} for x,y in zip(fpr,tpr)])
    ax.plot([0,1],[0,1],linestyle='--',color='black',linewidth=1.2)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate / Sensitivity'); ax.set_title(title,fontsize=16,weight='bold')
    ax.legend(fontsize=10); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(output_png,bbox_inches='tight')
    if metrics_png is not None:
        fig.savefig(metrics_png,bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(output_csv,index=False)


def save_pr_curves(y_true, prob, output_png, output_csv, title, metrics_png=None):
    rows=[]; fig,ax=plt.subplots(figsize=(9,7),dpi=400)
    for idx,label in enumerate(MAIN_LABEL_DISPLAY):
        binary=(y_true==idx).astype(int)
        if binary.min()==binary.max():
            continue
        precision,recall,_=precision_recall_curve(binary,prob[:,idx]); auprc_value=average_precision_score(binary,prob[:,idx])
        ax.plot(recall,precision,linewidth=2.2,label=f'{label} AUPRC={auprc_value:.3f}')
        rows.extend([{'label':label,'recall':float(r),'precision':float(p),'auprc':float(auprc_value)} for r,p in zip(recall,precision)])
    ax.set_xlabel('Recall / Sensitivity'); ax.set_ylabel('Precision'); ax.set_title(title,fontsize=16,weight='bold')
    ax.legend(fontsize=10); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(output_png,bbox_inches='tight')
    if metrics_png is not None:
        fig.savefig(metrics_png,bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame(rows).to_csv(output_csv,index=False)

def plot_training_curves(metrics_df, output_path, metrics_output_path=None):
    fig,axes=plt.subplots(1,2,figsize=(14,5),dpi=300)
    axes[0].plot(metrics_df['epoch'],metrics_df['train_loss'],label='train'); axes[0].plot(metrics_df['epoch'],metrics_df['val_loss'],label='val'); axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True,alpha=.3)
    axes[1].plot(metrics_df['epoch'],metrics_df['train_main_macro_f1'],label='train'); axes[1].plot(metrics_df['epoch'],metrics_df['val_main_macro_f1'],label='val'); axes[1].set_title('Macro F1'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True,alpha=.3)
    fig.tight_layout(); fig.savefig(output_path,bbox_inches='tight')
    if metrics_output_path is not None:
        fig.savefig(metrics_output_path,bbox_inches='tight')
    plt.close(fig)

def save_predictions(path, labels_df, indices, y_true, prob, pred):
    df=labels_df.iloc[indices].copy().reset_index(drop=True)
    df['true_class_index']=y_true.astype(int); df['pred_class_index']=pred.astype(int)
    df['true_label']=[MAIN_LABELS[int(i)] for i in y_true]; df['pred_label']=[MAIN_LABELS[int(i)] for i in pred]
    for i,name in enumerate(MAIN_LABELS): df[f'prob_{name}']=prob[:,i]
    df.to_csv(path,index=False)
    return df


## 9. 5-Fold Split Preparation


In [9]:
def make_cv_splits():
    y_dom=y_all[TRAINVAL_INDICES]
    if CONFIG['FAST_DEV_RUN']:
        skf=StratifiedKFold(n_splits=CONFIG['N_FOLDS'],shuffle=True,random_state=CONFIG['SEED'])
        tr_rel,va_rel=next(skf.split(TRAINVAL_INDICES,y_dom))
        return [{'fold':1,'train_idx':TRAINVAL_INDICES[tr_rel],'val_idx':TRAINVAL_INDICES[va_rel],'test_idx':TEST_INDICES.copy()}]
    skf=StratifiedKFold(n_splits=CONFIG['N_FOLDS'],shuffle=True,random_state=CONFIG['SEED'])
    return [{'fold':i,'train_idx':TRAINVAL_INDICES[tr],'val_idx':TRAINVAL_INDICES[va],'test_idx':TEST_INDICES.copy()} for i,(tr,va) in enumerate(skf.split(TRAINVAL_INDICES,y_dom),start=1)]
CV_SPLITS=make_cv_splits()
audit=[]
for s in CV_SPLITS:
    row={'fold':s['fold'],'train_beats':len(s['train_idx']),'val_beats':len(s['val_idx']),'test_beats':len(s['test_idx'])}
    for label_idx,label in enumerate(MAIN_LABELS):
        row[f'train_{label}']=int((y_all[s['train_idx']]==label_idx).sum()); row[f'val_{label}']=int((y_all[s['val_idx']]==label_idx).sum()); row[f'test_{label}']=int((y_all[s['test_idx']]==label_idx).sum())
    audit.append(row)
cv_audit=pd.DataFrame(audit); cv_audit.to_csv(AGG_DIR/'cv_split_audit.csv',index=False); display(cv_audit)


,fold,train_beats,val_beats,test_beats,train_Normal,val_Normal,test_Normal,train_Anterior,val_Anterior,test_Anterior,train_Inferior,val_Inferior,test_Inferior,train_Lateral,val_Lateral,test_Lateral
0,1,1555,389,499,502,125,135,323,81,97,606,152,184,124,31,83
1,2,1555,389,499,502,125,135,323,81,97,606,152,184,124,31,83
2,3,1555,389,499,502,125,135,323,81,97,606,152,184,124,31,83
3,4,1555,389,499,501,126,135,323,81,97,607,151,184,124,31,83
4,5,1556,388,499,501,126,135,324,80,97,607,151,184,124,31,83


## 10. Sanity Check Before Training


In [10]:
global_logger.info('Sanity check started.')
sample_split=CV_SPLITS[0]
train_loader,val_loader,test_loader=make_loaders(sample_split['train_idx'],sample_split['val_idx'],sample_split['test_idx'])
criterion=nn.CrossEntropyLoss().to(CONFIG['DEVICE'])
for strategy in (CONFIG['STRATEGIES'][:1] if CONFIG['FAST_DEV_RUN'] else CONFIG['STRATEGIES']):
    m=build_alpanet_model(); m=load_pretrained_weights(m,strategy,global_logger); m=apply_transfer_strategy(m,strategy); print_trainable_parameters(m,global_logger)
    b=next(iter(train_loader)); out=m(b['x'].to(CONFIG['DEVICE']))
    assert out['logits'].shape == (b['x'].shape[0], 4)
    assert out['probabilities'].shape == (b['x'].shape[0], 4)
    assert out['class_attention'].shape == (b['x'].shape[0], 4, 12)
    loss=criterion(out['logits'], b['y'].to(CONFIG['DEVICE'])); loss.backward()
    global_logger.info('Sanity passed strategy=%s loss=%.4f', strategy, float(loss.detach().cpu()))
    del m


2026-06-11 20:32:27,960 | INFO | Sanity check started.


2026-06-11 20:32:28,098 | INFO | No pretrained weights loaded.


2026-06-11 20:32:28,098 | INFO | Parameters trainable=512933 total=512933 ratio=1.0000


2026-06-11 20:32:28,423 | INFO | Sanity passed strategy=no_pretrain loss=1.3611


2026-06-11 20:32:28,449 | INFO | Loaded pretrained backbone.


...

2026-06-11 20:32:28,537 | INFO | Parameters trainable=448773 total=512933 ratio=0.8749


2026-06-11 20:32:28,607 | INFO | Sanity passed strategy=partial_finetune loss=1.4087


2026-06-11 20:32:28,635 | INFO | Loaded pretrained full model.


2026-06-11 20:32:28,636 | INFO | Parameters trainable=512933 total=512933 ratio=1.0000


2026-06-11 20:32:28,715 | INFO | Sanity passed strategy=full_finetune loss=0.7827


## 11. Run Strategy-First 5-Fold Training


In [11]:
def checkpoint_payload(model,optimizer,scheduler,fold,strategy,epoch,best_metric):
    return {
        'model_state_dict':model.state_dict(),
        'backbone_state_dict':model.backbone.state_dict(),
        'heads_state_dict':heads_state_dict(model),
        'optimizer_state_dict':optimizer.state_dict(),
        'scheduler_state_dict':scheduler.state_dict(),
        'strategy':strategy,
        'fold':fold,
        'epoch':epoch,
        'best_val_macro_f1':best_metric,
        'config':CONFIG,
        'pretrain_source':{k:str(v) for k,v in PRETRAIN_SOURCE.items()},
        'label_mapping':LABEL_MAPPING,
        'lead_prior_config':LEAD_PRIOR_CONFIG,
        'lead_order':LEAD_ORDER,
        'main_label_names':MAIN_LABELS,
        'main_label_display':MAIN_LABEL_DISPLAY,
        'main_class_prior_matrix':MAIN_CLASS_PRIOR_MATRIX.tolist(),
        'main_class_lead_priors':MAIN_CLASS_LEAD_PRIORS,
        'task_type':'single_label_4_class_softmax',
    }
def save_checkpoint(path,model,optimizer,scheduler,fold,strategy,epoch,best_metric):
    path.parent.mkdir(parents=True,exist_ok=True)
    torch.save(checkpoint_payload(model,optimizer,scheduler,fold,strategy,epoch,best_metric),path)

strategies = CONFIG['STRATEGIES'][:1] if CONFIG['FAST_DEV_RUN'] else CONFIG['STRATEGIES']
folds = CV_SPLITS[:1] if CONFIG['FAST_DEV_RUN'] else CV_SPLITS
all_results=[]
for strategy in strategies:
    global_logger.info('Strategy started: %s', strategy)
    for split in folds:
        fold=split['fold']; run_dir=STRATEGY_ROOT/strategy/f'fold_{fold}'
        for sub in ['checkpoints','checkpoints/per_5fold','logs','metrics','predictions','plots','configs']: (run_dir/sub).mkdir(parents=True,exist_ok=True)
        with open(run_dir/'configs'/'config.json','w') as f: json.dump({**CONFIG,'strategy':strategy,'fold':fold},f,indent=2)
        with open(run_dir/'configs'/'label_mapping.json','w') as f: json.dump(LABEL_MAPPING,f,indent=2)
        with open(run_dir/'configs'/'lead_prior_config.json','w') as f: json.dump(LEAD_PRIOR_CONFIG,f,indent=2)
        slogger=make_logger(f'ALPA-Net softmax {strategy} fold {fold}', run_dir/'logs'/'train.log', run_dir/'logs'/'error.log')
        result={'strategy':strategy,'fold':fold,'status':'FAILED','output_dir':str(run_dir)}
        try:
            fold_start=time.time(); set_seed(CONFIG['SEED']+fold)
            train_loader,val_loader,test_loader=make_loaders(split['train_idx'],split['val_idx'],split['test_idx'])
            model=build_alpanet_model(); model=load_pretrained_weights(model,strategy,slogger); model=apply_transfer_strategy(model,strategy); param_info=print_trainable_parameters(model,slogger)
            criterion=nn.CrossEntropyLoss().to(CONFIG['DEVICE'])
            optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=CONFIG['learning_rate'],weight_decay=CONFIG['weight_decay'])
            scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=CONFIG['scheduler_patience'])
            rows=[]; best_metric=-np.inf; best_val_loss=np.inf; patience=0
            epochs=2 if CONFIG['FAST_DEV_RUN'] else CONFIG['epochs']
            for epoch in range(1,epochs+1):
                t0=time.time(); tr=run_epoch(model,train_loader,criterion,optimizer); va=run_epoch(model,val_loader,criterion)
                tm,_,_,_=compute_metrics(tr['y_true'],tr['logits']); vm,_,_,_=compute_metrics(va['y_true'],va['logits'])
                scheduler.step(vm['macro_f1'])
                row={'strategy':strategy,'fold':fold,'epoch':epoch,'train_loss':tr['loss'],'val_loss':va['loss'],'train_main_macro_f1':tm['macro_f1'],'val_main_macro_f1':vm['macro_f1'],'train_accuracy':tm['accuracy'],'val_accuracy':vm['accuracy'],'train_balanced_accuracy':tm['balanced_accuracy'],'val_balanced_accuracy':vm['balanced_accuracy'],'learning_rate':optimizer.param_groups[0]['lr'],'epoch_time':time.time()-t0}
                for label in MAIN_LABELS: row[f'train_f1_{label}']=tm[f'f1_{label}']; row[f'val_f1_{label}']=vm[f'f1_{label}']
                rows.append(row); pd.DataFrame(rows).to_csv(run_dir/'metrics'/'metrics.csv',index=False)
                slogger.info('Epoch %03d/%03d train_loss=%.5f val_loss=%.5f train_macro_f1=%.4f val_macro_f1=%.4f', epoch, epochs, tr['loss'], va['loss'], tm['macro_f1'], vm['macro_f1'])
                save_checkpoint(run_dir/'checkpoints'/'last.pt',model,optimizer,scheduler,fold,strategy,epoch,best_metric)
                if vm['macro_f1'] > best_metric:
                    best_metric=vm['macro_f1']; patience=0; save_checkpoint(run_dir/'checkpoints'/'best_macro_f1.pt',model,optimizer,scheduler,fold,strategy,epoch,best_metric)
                else: patience += 1
                if va['loss'] < best_val_loss:
                    best_val_loss=va['loss']; save_checkpoint(run_dir/'checkpoints'/'best_val_loss.pt',model,optimizer,scheduler,fold,strategy,epoch,best_metric)
                if epoch >= 5 and epoch % 5 == 0:
                    periodic_path = run_dir/'checkpoints'/'per_5fold'/f'epoch_{epoch:03d}.pt'
                    save_checkpoint(periodic_path,model,optimizer,scheduler,fold,strategy,epoch,best_metric)
                    slogger.info('Saved periodic 5-epoch checkpoint: %s', periodic_path)
                if CONFIG['early_stopping_enabled'] and patience >= CONFIG['early_stopping_patience']:
                    slogger.info('Early stopping at epoch %d', epoch); break
            ckpt=torch.load(run_dir/'checkpoints'/'best_macro_f1.pt',map_location=CONFIG['DEVICE']); model.load_state_dict(ckpt['model_state_dict'])
            val_state=run_epoch(model,val_loader,criterion); test_state=run_epoch(model,test_loader,criterion)
            vm,val_prob,val_pred,val_class_metrics=compute_metrics(val_state['y_true'],val_state['logits']); testm,test_prob,test_pred,test_class_metrics=compute_metrics(test_state['y_true'],test_state['logits'])
            save_predictions(run_dir/'predictions'/'val_predictions.csv', labels_all, val_state['idx'], val_state['y_true'], val_prob, val_pred)
            save_predictions(run_dir/'predictions'/'test_predictions.csv', labels_all, test_state['idx'], test_state['y_true'], test_prob, test_pred)
            save_cm(val_state['y_true'],val_pred,MAIN_LABEL_DISPLAY,run_dir/'plots'/'val_confusion_matrix.png',run_dir/'plots'/'val_confusion_matrix.csv','Validation Confusion Matrix',metrics_dir=run_dir/'metrics')
            save_cm(test_state['y_true'],test_pred,MAIN_LABEL_DISPLAY,run_dir/'plots'/'test_confusion_matrix.png',run_dir/'plots'/'test_confusion_matrix.csv','Test Confusion Matrix',metrics_dir=run_dir/'metrics')
            save_roc_curves(val_state['y_true'],val_prob,run_dir/'plots'/'val_roc_curve.png',run_dir/'metrics'/'val_roc_curve.csv','Validation ROC Curve',metrics_png=run_dir/'metrics'/'val_roc_curve.png')
            save_roc_curves(test_state['y_true'],test_prob,run_dir/'plots'/'test_roc_curve.png',run_dir/'metrics'/'test_roc_curve.csv','Test ROC Curve',metrics_png=run_dir/'metrics'/'test_roc_curve.png')
            save_pr_curves(val_state['y_true'],val_prob,run_dir/'plots'/'val_pr_curve.png',run_dir/'metrics'/'val_pr_curve.csv','Validation PR Curve',metrics_png=run_dir/'metrics'/'val_pr_curve.png')
            save_pr_curves(test_state['y_true'],test_prob,run_dir/'plots'/'test_pr_curve.png',run_dir/'metrics'/'test_pr_curve.csv','Test PR Curve',metrics_png=run_dir/'metrics'/'test_pr_curve.png')
            plot_training_curves(pd.DataFrame(rows), run_dir/'plots'/'training_curve.png', metrics_output_path=run_dir/'metrics'/'training_curve.png')
            if 'class_attention' in test_state:
                mean_att=test_state['class_attention'].mean(axis=0); pd.DataFrame(MAIN_CLASS_PRIOR_MATRIX,index=MAIN_LABEL_DISPLAY,columns=LEAD_ORDER).to_csv(run_dir/'plots'/'initial_attention_prior_matrix.csv'); pd.DataFrame(MAIN_CLASS_PRIOR_MATRIX,index=MAIN_LABEL_DISPLAY,columns=LEAD_ORDER).to_csv(run_dir/'metrics'/'initial_attention_prior_matrix.csv'); pd.DataFrame(mean_att,index=MAIN_LABEL_DISPLAY,columns=LEAD_ORDER).to_csv(run_dir/'plots'/'lead_attention_heatmap.csv'); pd.DataFrame(mean_att,index=MAIN_LABEL_DISPLAY,columns=LEAD_ORDER).to_csv(run_dir/'metrics'/'lead_attention_heatmap.csv')
            test_f1_rows=[]
            val_f1_rows=[]
            for label,display_label in zip(MAIN_LABELS,MAIN_LABEL_DISPLAY):
                val_f1_rows.append({'split':'val','label':display_label,'internal_label':label,'f1_score':vm[f'f1_{label}']})
                test_f1_rows.append({'split':'test','label':display_label,'internal_label':label,'f1_score':testm[f'f1_{label}']})
            pd.DataFrame(test_f1_rows).to_csv(run_dir/'plots'/'test_f1_scores_by_class.csv',index=False)
            pd.DataFrame(test_f1_rows).to_csv(run_dir/'metrics'/'test_f1_scores_by_class.csv',index=False)
            val_class_metrics.insert(0,'split','val'); test_class_metrics.insert(0,'split','test')
            pd.concat([val_class_metrics,test_class_metrics],ignore_index=True).to_csv(run_dir/'metrics'/'per_class_metrics.csv',index=False)
            pd.concat([pd.DataFrame(val_f1_rows),pd.DataFrame(test_f1_rows)],ignore_index=True).to_csv(run_dir/'plots'/'f1_scores_by_class.csv',index=False)
            pd.concat([pd.DataFrame(val_f1_rows),pd.DataFrame(test_f1_rows)],ignore_index=True).to_csv(run_dir/'metrics'/'f1_scores_by_class.csv',index=False)
            pd.DataFrame([{'split':'val',**vm,'loss':val_state['loss']},{'split':'test',**testm,'loss':test_state['loss']}]).to_csv(run_dir/'metrics'/'final_metrics.csv',index=False)
            pd.DataFrame([{'split':'val',**vm,'loss':val_state['loss']},{'split':'test',**testm,'loss':test_state['loss']}]).to_csv(run_dir/'metrics'/'test_metrics.csv',index=False)
            pd.DataFrame([
                {'split':'val','macro_f1':vm['macro_f1'],'micro_f1':vm['micro_f1'],'weighted_f1':vm['weighted_f1'],'accuracy':vm['accuracy'],'loss':val_state['loss']},
                {'split':'test','macro_f1':testm['macro_f1'],'micro_f1':testm['micro_f1'],'weighted_f1':testm['weighted_f1'],'accuracy':testm['accuracy'],'loss':test_state['loss']},
            ]).to_csv(run_dir/'plots'/'f1_score_summary.csv',index=False)
            pd.DataFrame([
                {'split':'val','macro_f1':vm['macro_f1'],'micro_f1':vm['micro_f1'],'weighted_f1':vm['weighted_f1'],'balanced_accuracy':vm['balanced_accuracy'],'macro_auc':vm['macro_auc'],'accuracy':vm['accuracy'],'loss':val_state['loss']},
                {'split':'test','macro_f1':testm['macro_f1'],'micro_f1':testm['micro_f1'],'weighted_f1':testm['weighted_f1'],'balanced_accuracy':testm['balanced_accuracy'],'macro_auc':testm['macro_auc'],'accuracy':testm['accuracy'],'loss':test_state['loss']},
            ]).to_csv(run_dir/'metrics'/'f1_score_summary.csv',index=False)
            with open(run_dir/'metrics'/'test_metrics.json','w') as f: json.dump({'val':vm,'test':testm,'val_loss':val_state['loss'],'test_loss':test_state['loss']},f,indent=2)
            result={'strategy':strategy,'fold':fold,'status':'OK','error':'','train_main_macro_f1':rows[-1]['train_main_macro_f1'],'val_main_macro_f1':vm['macro_f1'],'test_main_macro_f1':testm['macro_f1'],'test_main_micro_f1':testm['micro_f1'],'test_main_weighted_f1':testm['weighted_f1'],'test_accuracy':testm['accuracy'],'test_balanced_accuracy':testm['balanced_accuracy'],'test_macro_auc':testm['macro_auc'],'test_loss':test_state['loss'],'checkpoint_path':str(run_dir/'checkpoints'/'best_macro_f1.pt'),'output_dir':str(run_dir),**param_info}
            for label,display_label in zip(MAIN_LABELS,MAIN_LABEL_DISPLAY):
                result[f'train_f1_{display_label}']=rows[-1].get(f'train_f1_{label}',np.nan)
                result[f'val_f1_{display_label}']=vm[f'f1_{label}']
                result[f'test_f1_{display_label}']=testm[f'f1_{label}']
                result[f'test_sensitivity_{display_label}']=testm[f'sensitivity_{label}']
                result[f'test_specificity_{display_label}']=testm[f'specificity_{label}']
                result[f'test_auc_{display_label}']=testm[f'auc_{label}']
            slogger.info('Fold completed strategy=%s fold=%s test_macro_f1=%.4f duration=%.1fs', strategy, fold, testm['macro_f1'], time.time()-fold_start)
        except Exception as exc:
            result.update({'status':'FAILED','error':str(exc)})
            slogger.error('FAILED strategy=%s fold=%s: %s\n%s',strategy,fold,exc,traceback.format_exc())
        all_results.append(result); pd.DataFrame(all_results).to_csv(AGG_DIR/'all_fold_metrics.csv',index=False); global_logger.info('Finished strategy=%s fold=%s status=%s',strategy,fold,result['status'])


2026-06-11 20:32:28,832 | INFO | Strategy started: no_pretrain


2026-06-11 20:32:28,841 | INFO | No pretrained weights loaded.


2026-06-11 20:32:28,842 | INFO | Parameters trainable=512933 total=512933 ratio=1.0000


2026-06-11 20:32:30,220 | INFO | Epoch 001/050 train_loss=1.23166 val_loss=1.18544 train_macro_f1=0.2538 val_macro_f1=0.2142


2026-06-11 20:32:30,864 | INFO | Epoch 002/050 train_loss=1.00640 val_loss=0.85033 train_macro_f1=0.4167 val_macro_f1=0.4823


...

2026-06-11 20:47:26,652 | INFO | Epoch 049/050 train_loss=0.00081 val_loss=0.02692 train_macro_f1=1.0000 val_macro_f1=0.9964


2026-06-11 20:47:27,869 | INFO | Epoch 050/050 train_loss=0.00058 val_loss=0.02872 train_macro_f1=1.0000 val_macro_f1=0.9945


2026-06-11 20:47:27,900 | INFO | Saved periodic 5-epoch checkpoint: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/5_alpanet_finetune_no_attention/20260611_203227/transfer_strategies/full_finetune/fold_5/checkpoints/per_5fold/epoch_050.pt


2026-06-11 20:47:32,608 | INFO | Fold completed strategy=full_finetune fold=5 test_macro_f1=0.9074 duration=64.6s


2026-06-11 20:47:32,611 | INFO | Finished strategy=full_finetune fold=5 status=OK


## 12. Aggregate Results and Plots


In [12]:
all_fold_metrics=pd.read_csv(AGG_DIR/'all_fold_metrics.csv')
ok=all_fold_metrics[all_fold_metrics['status'].eq('OK')].copy()
if ok.empty:
    raise RuntimeError('No successful fold results to aggregate.')

f1_detail_cols = [f'{split}_f1_{label}' for split in ['train','val','test'] for label in MAIN_LABEL_DISPLAY]
per_class_extra_cols = [f'test_{metric}_{label}' for metric in ['sensitivity','specificity','auc'] for label in MAIN_LABEL_DISPLAY]

summary_metrics = [
    'test_main_macro_f1', 'test_main_micro_f1', 'test_main_weighted_f1',
    'test_accuracy', 'test_balanced_accuracy', 'test_macro_auc', 'test_loss',
    *[f'test_f1_{label}' for label in MAIN_LABEL_DISPLAY],
    *per_class_extra_cols,
]
summary_metrics = [col for col in summary_metrics if col in ok.columns]

avg = ok.groupby('strategy').agg(
    avg_test_main_macro_f1=('test_main_macro_f1','mean'),
    std_test_main_macro_f1=('test_main_macro_f1','std'),
    avg_test_main_micro_f1=('test_main_micro_f1','mean'),
    std_test_main_micro_f1=('test_main_micro_f1','std'),
    avg_test_main_weighted_f1=('test_main_weighted_f1','mean'),
    std_test_main_weighted_f1=('test_main_weighted_f1','std'),
    avg_test_accuracy=('test_accuracy','mean'),
    std_test_accuracy=('test_accuracy','std'),
    avg_test_balanced_accuracy=('test_balanced_accuracy','mean'),
    std_test_balanced_accuracy=('test_balanced_accuracy','std'),
    avg_test_macro_auc=('test_macro_auc','mean'),
    std_test_macro_auc=('test_macro_auc','std'),
    avg_test_loss=('test_loss','mean'),
    std_test_loss=('test_loss','std'),
).reset_index()
best_by_strategy=ok.sort_values('test_main_macro_f1',ascending=False).groupby('strategy').first().reset_index()[['strategy','fold']].rename(columns={'fold':'best_fold_for_strategy'})
avg=avg.merge(best_by_strategy,on='strategy',how='left').sort_values('avg_test_main_macro_f1',ascending=False)

mean_std_rows=[]
for strategy, part in ok.groupby('strategy'):
    row={'strategy':strategy,'n_folds':int(len(part))}
    for metric in summary_metrics:
        mean=float(part[metric].mean())
        std=float(part[metric].std(ddof=1)) if len(part)>1 else 0.0
        row[f'{metric}_mean']=mean
        row[f'{metric}_std']=std
        row[f'{metric}_mean_std']=f'{mean:.4f} ± {std:.4f}'
    mean_std_rows.append(row)
mean_std=pd.DataFrame(mean_std_rows).sort_values('test_main_macro_f1_mean',ascending=False)

rank=ok.sort_values('test_main_macro_f1',ascending=False).reset_index(drop=True)
rank.insert(0,'rank',np.arange(1,len(rank)+1))
per_class_cols=[c for c in ['strategy','fold',*f1_detail_cols,*per_class_extra_cols] if c in all_fold_metrics.columns]
detailed_cols=[c for c in ['strategy','fold','train_main_macro_f1','val_main_macro_f1','test_main_macro_f1','test_main_micro_f1','test_main_weighted_f1','test_accuracy','test_balanced_accuracy','test_macro_auc','test_loss','trainable_params','total_params','trainable_ratio','status','checkpoint_path','output_dir',*f1_detail_cols,*per_class_extra_cols] if c in all_fold_metrics.columns]

# Save in both aggregate_results and root metrics for paper/table access.
for out_dir in [AGG_DIR, ROOT_METRICS_DIR]:
    avg.to_csv(out_dir/'average_metrics_by_strategy.csv',index=False)
    mean_std.to_csv(out_dir/'cross_validation_mean_std_by_strategy.csv',index=False)
    rank.to_csv(out_dir/'best_fold_ranking.csv',index=False)
    all_fold_metrics[detailed_cols].to_csv(out_dir/'detailed_fold_metrics.csv',index=False)
    all_fold_metrics[per_class_cols].to_csv(out_dir/'per_class_f1_by_strategy_fold.csv',index=False)

with pd.ExcelWriter(ROOT_METRICS_DIR/'alpanet_transfer_summary.xlsx') as writer:
    avg.to_excel(writer,sheet_name='avg_per_strategy',index=False)
    mean_std.to_excel(writer,sheet_name='mean_std_per_strategy',index=False)
    rank.to_excel(writer,sheet_name='best_fold_ranking',index=False)
    all_fold_metrics[detailed_cols].to_excel(writer,sheet_name='detailed_per_strategy_fold',index=False)
with pd.ExcelWriter(AGG_DIR/'alpanet_transfer_summary.xlsx') as writer:
    avg.to_excel(writer,sheet_name='avg_per_strategy',index=False)
    mean_std.to_excel(writer,sheet_name='mean_std_per_strategy',index=False)
    rank.to_excel(writer,sheet_name='best_fold_ranking',index=False)
    all_fold_metrics[detailed_cols].to_excel(writer,sheet_name='detailed_per_strategy_fold',index=False)

rank['strategy_fold']=rank['strategy']+'_fold_'+rank['fold'].astype(str)
plt.figure(figsize=(16,6)); sns.barplot(data=rank,x='strategy_fold',y='test_main_macro_f1',color='#d55e00'); plt.xticks(rotation=60,ha='right'); plt.title('All Strategy-Fold Test Macro-F1 Ranking'); plt.tight_layout(); plt.savefig(PLOTS_DIR/'all_strategy_fold_test_macro_f1_ranking.png',dpi=400,bbox_inches='tight'); plt.close()
plt.figure(figsize=(10,max(6,0.35*len(rank)))); sns.barplot(data=rank,y='strategy_fold',x='test_main_macro_f1',color='#d55e00'); plt.title('All Strategy-Fold Test Macro-F1 Ranking'); plt.tight_layout(); plt.savefig(PLOTS_DIR/'all_strategy_fold_test_macro_f1_ranking_horizontal.png',dpi=400,bbox_inches='tight'); plt.close()
plt.figure(figsize=(9,6)); sns.barplot(data=avg,x='strategy',y='avg_test_main_macro_f1',color='#d55e00'); plt.xticks(rotation=20); plt.title('Average Test Macro-F1 by Strategy'); plt.tight_layout(); plt.savefig(PLOTS_DIR/'avg_test_macro_f1_by_strategy.png',dpi=400,bbox_inches='tight'); plt.close()
plt.figure(figsize=(9,6)); sns.boxplot(data=ok,x='strategy',y='test_main_macro_f1',color='#f6ad55'); sns.stripplot(data=ok,x='strategy',y='test_main_macro_f1',color='black',alpha=.65); plt.xticks(rotation=20); plt.title('Foldwise Test Macro-F1 by Strategy'); plt.tight_layout(); plt.savefig(PLOTS_DIR/'foldwise_test_macro_f1_boxplot_by_strategy.png',dpi=400,bbox_inches='tight'); plt.close()
display(avg); display(mean_std); display(rank.head(10))
RUN_FINISHED_AT=datetime.now(); RUN_DURATION_SECONDS=time.time()-RUN_START_TIME
global_logger.info('Run finished at: %s', RUN_FINISHED_AT.strftime('%Y-%m-%d %H:%M:%S'))
global_logger.info('Total duration %.2f seconds', RUN_DURATION_SECONDS)
print('Notebook complete. Output:', OUTPUT_DIR)


,strategy,avg_test_main_macro_f1,std_test_main_macro_f1,avg_test_main_micro_f1,std_test_main_micro_f1,avg_test_main_weighted_f1,std_test_main_weighted_f1,avg_test_accuracy,std_test_accuracy,avg_test_balanced_accuracy,std_test_balanced_accuracy,avg_test_macro_auc,std_test_macro_auc,avg_test_loss,std_test_loss,best_fold_for_strategy
3,partial_finetune,0.878945,0.030881,0.873347,0.032295,0.872709,0.031740,0.873347,0.032295,0.869353,0.035797,0.980149,0.007008,0.435719,0.114978,2
1,full_finetune,0.874586,0.042088,0.878958,0.033997,0.877543,0.035540,0.878958,0.033997,0.866868,0.049535,0.985969,0.004564,0.414004,0.069521,2
2,no_pretrain,0.866505,0.063330,0.865331,0.052725,0.865678,0.053094,0.865331,0.052725,0.861610,0.069540,0.973993,0.008048,0.557973,0.164656,5
0,frozen_backbone,0.709164,0.040431,0.758717,0.018014,0.739518,0.028443,0.758717,0.018014,0.703531,0.031593,0.962713,0.001189,0.558541,0.032375,2


,strategy,n_folds,test_main_macro_f1_mean,test_main_macro_f1_std,test_main_macro_f1_mean_std,test_main_micro_f1_mean,test_main_micro_f1_std,test_main_micro_f1_mean_std,test_main_weighted_f1_mean,test_main_weighted_f1_std,...,test_auc_NORM_mean_std,test_auc_AMI_mean,test_auc_AMI_std,test_auc_AMI_mean_std,test_auc_IMI_mean,test_auc_IMI_std,test_auc_IMI_mean_std,test_auc_LMI_mean,test_auc_LMI_std,test_auc_LMI_mean_std
3,partial_finetune,5,0.878945,0.030881,0.8789 ± 0.0309,0.873347,0.032295,0.8733 ± 0.0323,0.872709,0.031740,...,0.9910 ± 0.0053,0.973227,0.014810,0.9732 ± 0.0148,0.956339,0.013310,0.9563 ± 0.0133,0.999983,0.000026,1.0000 ± 0.0000
1,full_finetune,5,0.874586,0.042088,0.8746 ± 0.0421,0.878958,0.033997,0.8790 ± 0.0340,0.877543,0.035540,...,0.9937 ± 0.0050,0.976186,0.006482,0.9762 ± 0.0065,0.974120,0.009072,0.9741 ± 0.0091,0.999832,0.000232,0.9998 ± 0.0002
2,no_pretrain,5,0.866505,0.063330,0.8665 ± 0.0633,0.865331,0.052725,0.8653 ± 0.0527,0.865678,0.053094,...,0.9975 ± 0.0021,0.941981,0.020511,0.9420 ± 0.0205,0.956482,0.019940,0.9565 ± 0.0199,0.999988,0.000026,1.0000 ± 0.0000
0,frozen_backbone,5,0.709164,0.040431,0.7092 ± 0.0404,0.758717,0.018014,0.7587 ± 0.0180,0.739518,0.028443,...,0.9570 ± 0.0057,0.973027,0.001588,0.9730 ± 0.0016,0.920914,0.005939,0.9209 ± 0.0059,0.999913,0.000130,0.9999 ± 0.0001


,rank,strategy,fold,status,error,train_main_macro_f1,val_main_macro_f1,test_main_macro_f1,test_main_micro_f1,test_main_weighted_f1,...,test_sensitivity_IMI,test_specificity_IMI,test_auc_IMI,train_f1_LMI,val_f1_LMI,test_f1_LMI,test_sensitivity_LMI,test_specificity_LMI,test_auc_LMI,strategy_fold
0,1,no_pretrain,5,OK,NaN,0.998633,0.990868,0.921050,0.917836,0.916537,...,0.961957,0.892063,0.982574,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,no_pretrain_fold_5
1,2,full_finetune,2,OK,NaN,0.998747,0.998176,0.918105,0.915832,0.915877,...,0.869565,0.955556,0.972257,0.995984,1.000000,1.000000,1.000000,1.000000,1.000000,full_finetune_fold_2
2,3,partial_finetune,2,OK,NaN,0.997586,0.985613,0.908475,0.907816,0.906888,...,0.940217,0.898413,0.977312,1.000000,1.000000,0.949367,0.903614,1.000000,0.999971,partial_finetune_fold_2
3,4,full_finetune,5,OK,NaN,1.000000,0.998179,0.907363,0.905812,0.905142,...,0.929348,0.895238,0.986111,1.000000,1.000000,0.994012,1.000000,0.997596,1.000000,full_finetune_fold_5
4,5,partial_finetune,3,OK,NaN,0.995419,1.000000,0.900986,0.895792,0.895198,...,0.864130,0.920635,0.960455,0.995984,1.000000,0.981595,0.963855,1.000000,1.000000,partial_finetune_fold_3
5,6,no_pretrain,4,OK,NaN,0.998790,0.998179,0.893742,0.887776,0.889710,...,0.815217,0.936508,0.926691,0.995984,1.000000,0.987805,0.975904,1.000000,1.000000,no_pretrain_fold_4
6,7,full_finetune,3,OK,NaN,0.998952,0.996350,0.886403,0.885772,0.885569,...,0.891304,0.904762,0.964044,1.000000,1.000000,0.957055,0.939759,0.995192,0.999623,full_finetune_fold_3
7,8,no_pretrain,3,OK,NaN,0.998292,1.000000,0.885645,0.877756,0.877116,...,0.842391,0.898413,0.955305,0.995984,1.000000,1.000000,1.000000,1.000000,1.000000,no_pretrain_fold_3
8,9,partial_finetune,4,OK,NaN,0.999407,1.000000,0.885563,0.877756,0.876435,...,0.842391,0.901587,0.951536,1.000000,1.000000,0.993939,0.987952,1.000000,1.000000,partial_finetune_fold_4
9,10,no_pretrain,1,OK,NaN,0.999089,0.993207,0.874639,0.865731,0.868634,...,0.809783,0.907937,0.956815,1.000000,0.984127,0.994012,1.000000,0.997596,1.000000,no_pretrain_fold_1


2026-06-11 20:47:34,136 | INFO | Run finished at: 2026-06-11 20:47:34


2026-06-11 20:47:34,137 | INFO | Total duration 906.27 seconds


Notebook complete. Output: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-localization/outputs/5_alpanet_finetune_no_attention/20260611_203227
